# MoE（专家混合）架构
## Mixture of Experts Architecture

<img src="../images/logo.png" width=150>

MoE通过稀疏激活机制，在保持参数总量不变的情况下大幅增加模型容量。Switch Transformer使用稀疏门控，每个token只激活少数专家，大大减少了计算量。

MoE uses sparse activation to dramatically increase model capacity while keeping total parameters constant. Switch Transformer uses sparse gating where each token activates only a few experts, greatly reducing computation.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

class Expert(nn.Module):
    """
    单个专家网络
    Single expert network (Feed-Forward Network)
    """
    def __init__(self, input_dim, expert_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, expert_dim),
            nn.GELU(),
            nn.Linear(expert_dim, input_dim)
        )
    
    def forward(self, x):
        return self.net(x)

class MoELayer(nn.Module):
    """
    MoE层：多个专家 + 路由机制
    MoE Layer: Multiple experts + Routing mechanism
    """
    def __init__(self, embed_dim, num_experts=8, expert_dim=2048, top_k=2):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        
        # 专家网络 / Expert networks
        self.experts = nn.ModuleList([
            Expert(embed_dim, expert_dim)
            for _ in range(num_experts)
        ])
        
        # 路由网络 / Router network
        self.router = nn.Linear(embed_dim, num_experts, bias=False)
    
    def forward(self, x):
        """x: (batch, seq_len, embed_dim)"""
        B, N, C = x.shape
        
        # 计算路由logits / Compute router logits
        router_logits = self.router(x)  # (B, N, num_experts)
        router_probs = F.softmax(router_logits, dim=-1)
        
        # 选择top-k专家 / Select top-k experts
        top_k_probs, top_k_indices = torch.topk(router_probs, self.top_k, dim=-1)
        top_k_probs = top_k_probs / top_k_probs.sum(dim=-1, keepdim=True)  # 归一化 / Normalize
        
        # 初始化输出 / Initialize output
        output = torch.zeros_like(x)
        
        # 逐个处理top-k专家 / Process each top-k expert
        for k_idx in range(self.top_k):
            expert_weights = top_k_probs[:, :, k_idx].unsqueeze(-1)  # (B, N, 1)
            expert_indices = top_k_indices[:, :, k_idx]  # (B, N)
            
            # 聚合专家输出 / Aggregate expert outputs
            for b in range(B):
                for n in range(N):
                    expert_id = expert_indices[b, n].item()
                    output[b, n] += expert_weights[b, n, 0] * self.experts[expert_id](x[b, n])
        
        return output

# 测试 / Test
moe_layer = MoELayer(embed_dim=64, num_experts=8, expert_dim=256, top_k=2)
x = torch.randn(2, 16, 64)
output = moe_layer(x)
print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Number of experts: {moe_layer.num_experts}")
print(f"Top-k: {moe_layer.top_k}")

# 门控机制可视化
## Gating Mechanism Visualization

In [ ]:
# 可视化路由分布
# Visualize routing distribution

num_experts = 8
top_k = 2
num_tokens = 100

# 模拟路由器输出 / Simulate router output
router_logits = torch.randn(num_tokens, num_experts)
router_probs = F.softmax(router_logits, dim=-1)
top_k_probs, top_k_indices = torch.topk(router_probs, top_k, dim=-1)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Router概率分布 / Router probability distribution
im = axes[0].imshow(router_probs[:30].T, aspect='auto', cmap='viridis')
axes[0].set_title('Router Probability Distribution\n(first 30 tokens)')
axes[0].set_xlabel('Token')
axes[0].set_ylabel('Expert')
plt.colorbar(im, ax=axes[0])

# 每个token被激活的专家频率 / Expert activation frequency per token
expert_counts = torch.zeros(num_experts)
for indices in top_k_indices:
    for idx in indices:
        expert_counts[idx.item()] += 1

axes[1].bar(range(num_experts), expert_counts.numpy())
axes[1].set_title('Expert Activation Frequency')
axes[1].set_xlabel('Expert ID')
axes[1].set_ylabel('Activation Count')
axes[1].set_xticks(range(num_experts))

plt.tight_layout()
plt.savefig('../images/moe_routing.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nTotal activations: {expert_counts.sum().item():.0f}")
print(f"Load balancing: {expert_counts.std().item() / expert_counts.mean().item():.2f} (lower is better)")

# DeepSeekMoE架构
## DeepSeekMoE Architecture

In [ ]:
class DeepSeekMoELayer(nn.Module):
    """
    DeepSeekMoE的关键特性：
    1. 细粒度专家分割（将专家分成更多小专家）
    2. 共享专家隔离（某些专家被所有token共享）
    3. 设备限制路由（分布式训练优化）
    
    DeepSeekMoE key features:
    1. Fine-grained expert splitting
    2. Shared expert isolation
    3. Device-limited routing
    """
    def __init__(self, embed_dim, num_experts=8, expert_dim=2048, top_k=2, num_shared=2):
        super().__init__()
        self.num_experts = num_experts
        self.num_shared = num_shared
        self.top_k = top_k
        
        # 共享专家（始终激活）/ Shared experts (always activated)
        self.shared_experts = nn.ModuleList([
            Expert(embed_dim, expert_dim)
            for _ in range(num_shared)
        ])
        
        # 路由专家（稀疏激活）/ Routed experts (sparsely activated)
        self.routed_experts = nn.ModuleList([
            Expert(embed_dim, expert_dim)
            for _ in range(num_experts)
        ])
        
        # 路由器 / Router
        self.router = nn.Linear(embed_dim, num_experts, bias=False)
        
        # 专家分发权重（用于设备限制）/ Expert dispatch weights
        self.expert_assignment = None  # 简化为无设备限制 / Simplified without device limit
    
    def forward(self, x):
        B, N, C = x.shape
        
        # 1. 共享专家输出（始终计算）/ Shared expert output (always computed)
        shared_output = sum([expert(x) for expert in self.shared_experts])
        
        # 2. 路由专家输出（稀疏激活）/ Routed expert output (sparse activation)
        router_logits = self.router(x)
        router_probs = F.softmax(router_logits, dim=-1)
        top_k_probs, top_k_indices = torch.topk(router_probs, self.top_k, dim=-1)
        top_k_probs = top_k_probs / top_k_probs.sum(dim=-1, keepdim=True)
        
        routed_output = torch.zeros_like(x)
        for k_idx in range(self.top_k):
            expert_weights = top_k_probs[:, :, k_idx].unsqueeze(-1)
            expert_indices = top_k_indices[:, :, k_idx]
            
            for b in range(B):
                for n in range(N):
                    expert_id = expert_indices[b, n].item()
                    routed_output[b, n] += expert_weights[b, n, 0] * self.routed_experts[expert_id](x[b, n])
        
        return shared_output + routed_output

# 测试 / Test
deepseek_moe = DeepSeekMoELayer(embed_dim=64, num_experts=8, expert_dim=256, top_k=2, num_shared=2)
x = torch.randn(2, 16, 64)
output = deepseek_moe(x)
print(f"DeepSeekMoE input: {x.shape} -> output: {output.shape}")
print(f"Shared experts: {deepseek_moe.num_shared}")
print(f"Routed experts: {deepseek_moe.num_experts}")

In [ ]:
# MoE负载均衡可视化 / MoE Load Balancing Visualization
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 1. Load imbalance without balancing / 无负载均衡
ax1 = axes[0]
num_experts = 8
unbalanced_load = [300, 50, 40, 30, 25, 20, 20, 15]
ax1.bar(range(num_experts), unbalanced_load, color='#e74c3c', edgecolor='black')
ax1.set_xlabel('Expert ID')
ax1.set_ylabel('Tokens Assigned')
ax1.set_title('Poor Load Balancing
(Expert 0 handles 60% of tokens)')
ax1.axhline(y=np.mean(unbalanced_load), color='gray', linestyle='--', label='Ideal')

# 2. Load with auxiliary loss / 辅助损失下的负载均衡
ax2 = axes[1]
balanced_load = [130, 125, 128, 135, 122, 130, 127, 103]
ax2.bar(range(num_experts), balanced_load, color='#2ecc71', edgecolor='black')
ax2.set_xlabel('Expert ID')
ax2.set_ylabel('Tokens Assigned')
ax2.set_title('With Load Balancing Loss
(More even distribution)')
ax2.axhline(y=np.mean(balanced_load), color='gray', linestyle='--', label='Ideal')

# 3. DeepSeek shared + routed experts
ax3 = axes[2]
expert_types = ['Shared
Expert 1', 'Shared
Expert 2', 'Routed
Expert 1', 'Routed
Expert 2', 
                'Routed
Expert 3', 'Routed
Expert 4', 'Routed
Expert 5', 'Routed
Expert 6']
activation = [1.0, 1.0, 0.45, 0.52, 0.38, 0.41, 0.55, 0.49]
colors = ['#3498db']*2 + ['#e74c3c']*6

bars = ax3.bar(expert_types, activation, color=colors)
ax3.set_ylabel('Activation Probability')
ax3.set_title('DeepSeekMoE: Shared + Routed Experts
(Shared always active)')
ax3.tick_params(axis='x', rotation=45)

# Add legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#3498db', label='Shared (always active)'),
                  Patch(facecolor='#e74c3c', label='Routed (selective)')]
ax3.legend(handles=legend_elements)

plt.tight_layout()
plt.savefig('../images/moe_load_balancing.png', dpi=150, bbox_inches='tight')
plt.show()

print("MoE load balancing visualization saved!")

# 参数量与计算量对比
## Parameters vs Computation Comparison

In [ ]:
# 对比Dense模型和MoE模型的参数量和计算量
# Compare dense vs MoE model parameters and computation

embed_dim = 512
ff_dim = 2048
num_layers = 12

# Dense模型 / Dense model
dense_params = num_layers * (2 * embed_dim * ff_dim + 2 * ff_dim * embed_dim)
dense_flops = num_layers * 2 * embed_dim * ff_dim  # 每次前向 / Per forward pass

# MoE模型 / MoE model
num_experts_list = [8, 16, 32, 64]
top_k = 2

moe_params_list = []
moe_flops_list = []

for num_experts in num_experts_list:
    # 总参数 = 共享FFN + 所有专家参数
    # Total params = shared FFN + all expert params
    moe_params = num_layers * (2 * embed_dim * ff_dim + num_experts * 2 * embed_dim * ff_dim)
    # 每次前向只激活top_k个专家
    # Per forward only activates top_k experts
    moe_flops = num_layers * (2 * embed_dim * ff_dim + top_k * 2 * embed_dim * ff_dim)
    moe_params_list.append(moe_params)
    moe_flops_list.append(moe_flops)

# 可视化 / Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 参数量对比 / Parameters comparison
axes[0].bar(['Dense'] + [f'MoE-{n}' for n in num_experts_list], 
            [dense_params] + moe_params_list, color=['blue'] + ['orange']*4)
axes[0].set_title('Total Parameters')
axes[0].set_ylabel('Parameters')
axes[0].ticklabel_format(style='scientific', axis='y', scilimits=(0,0))

# 计算量对比 / FLOPs comparison
axes[1].bar(['Dense'] + [f'MoE-{n}' for n in num_experts_list],
            [dense_flops] + moe_flops_list, color=['blue'] + ['orange']*4)
axes[1].set_title('FLOPs per Forward Pass')
axes[1].set_ylabel('FLOPs')
axes[1].ticklabel_format(style='scientific', axis='y', scilimits=(0,0))

plt.tight_layout()
plt.savefig('../images/moe_params_flops.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nDense model parameters: {dense_params:,}")
print(f"Dense model FLOPs: {dense_flops:,}")
for n, p, f in zip(num_experts_list, moe_params_list, moe_flops_list):
    print(f"MoE-{n}: params={p:,}, FLOPs={f:,}, ratio={f/dense_flops:.2f}x")

# 整合到完整Transformer
## Integration into Complete Transformer

In [ ]:
class MoETransformerBlock(nn.Module):
    """MoE Transformer块"""
    
    def __init__(self, embed_dim, num_heads, num_experts, expert_dim, top_k, ff_dim):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.attention = nn.MultiheadAttention(embed_dim, num_heads, dropout=0.1, batch_first=True)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.moe = MoELayer(embed_dim, num_experts, expert_dim, top_k)
    
    def forward(self, x, mask=None):
        # 自注意力 / Self-attention
        x = x + self.attention(self.ln1(x), self.ln1(x), self.ln1(x), attn_mask=mask)[0]
        # MoE前馈 / MoE feed-forward
        x = x + self.moe(self.ln2(x))
        return x

# 创建MoE模型 / Create MoE model
def create_moe_transformer(num_layers, embed_dim, num_heads, num_experts, expert_dim, top_k):
    ff_dim = expert_dim  # 前馈维度等于专家维度 / FF dimension equals expert dimension
    
    return nn.Sequential(*[
        MoETransformerBlock(embed_dim, num_heads, num_experts, expert_dim, top_k, ff_dim)
        for _ in range(num_layers)
    ])

# 测试 / Test
moe_model = create_moe_transformer(
    num_layers=4,
    embed_dim=256,
    num_heads=8,
    num_experts=8,
    expert_dim=1024,
    top_k=2
)

x = torch.randn(2, 32, 256)
output = moe_model(x)

total_params = sum(p.numel() for p in moe_model.parameters())
print(f"MoE Transformer model:")
print(f"  Input shape: {x.shape}")
print(f"  Output shape: {output.shape}")
print(f"  Total parameters: {total_params:,}")
print(f"  Parameters per layer: {total_params // 4:,}")

# 总结

| 特性 | Dense | MoE |
|------|-------|-----|
| 参数总量 | 固定 | 可大幅增加 |
| 每次前向计算 | 全部激活 | 只激活top-k |
| 计算效率 | O(n) | O(n×k/N) |
| 内存占用 | 均匀 | 分布不均 |

MoE是现代超大模型（如Mixtral、DeepSeek）的核心技术，通过稀疏激活实现高效的大模型训练和推理。

# 已实现 / Implemented

本notebook已完整实现以下内容：

1. **基础MoE层** - 标准专家混合层，top-k路由
2. **门控机制可视化** - 路由器概率分布和专家激活频率
3. **DeepSeekMoE架构** - 共享专家 + 细粒度路由专家
4. **参数量与计算量对比** - Dense vs MoE的FLOPs分析
5. **完整MoE Transformer** - 整合到Transformer块

## 扩展阅读 / Further Reading

| 主题 | 说明 | 推荐资源 |
|------|------|----------|
| **Expert Parallelism** | 专家并行分布式训练 | [Megatron-M](https://arxiv.org/abs/1909.08053) |
| **Load Balancing Loss** | 辅助损失实现负载均衡 | [Switch Transformer](https://arxiv.org/abs/2101.03961) |
| **Auxiliary-loss-free** | DeepSeek的无辅助损失负载均衡 | [DeepSeek-MoE](https://arxiv.org/abs/2401.14166) |
| **Expert Routing** | 动态专家选择策略 | [GShard](https://arxiv.org/abs/2006.16668) |
| **Mixtral 8x7B** | 开源MoE模型实践 | [Mixtral Paper](https://arxiv.org/abs/2401.04088) |
